# Predicting Tax Refund or Tax Owed Outcomes Using Financial Activity Patterns
**Adhilingam Adiswaran - DSC 680 Milestone 2**

This notebook generates the privacy-preserving synthetic dataset, performs reproducible data preparation and feature engineering, trains Logistic Regression, Decision Tree, and Random Forest classifiers, validates them with a held-out test set and five-fold stratified cross-validation, and evaluates subgroup performance. The synthetic design follows the Milestone 1 proposal and is an educational prototype, not tax advice.

## Method justification
Logistic Regression is used as an interpretable baseline; a Decision Tree provides readable nonlinear rules; and Random Forest captures nonlinear relationships and interactions while reducing the instability of a single tree. Accuracy is not used alone because the synthetic outcome is imbalanced. Precision, recall, F1, ROC-AUC, confusion matrices, cross-validation, permutation importance, and subgroup metrics are therefore reported.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, json, os
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, RocCurveDisplay
from sklearn.inspection import permutation_importance

OUT='tax_project_assets'; os.makedirs(OUT, exist_ok=True)
rng=np.random.default_rng(680)
n=5000
filing=rng.choice(['Single','Married Filing Jointly','Head of Household'], n, p=[.48,.38,.14])
base=rng.lognormal(mean=np.log(68000), sigma=.62, size=n)
wages=np.clip(base, 12000, 350000)
other=np.where(rng.random(n)<.28, rng.gamma(2,3500,n), 0)
deps=np.where(filing=='Single', rng.choice([0,1,2],n,p=[.82,.13,.05]), rng.choice([0,1,2,3],n,p=[.28,.32,.27,.13]))
agi=wages+other-rng.uniform(0,3500,n)
itemized=(rng.random(n)<np.clip(.10 + agi/700000, .10,.42)).astype(int)
deduction=np.where(itemized, np.clip(rng.normal(26000,9000,n),13000,60000), np.where(filing=='Married Filing Jointly',29200,np.where(filing=='Head of Household',21900,14600)))
taxable=np.maximum(0,agi-deduction)
# simplified effective tax function for synthetic study, not tax calculation
rate=np.select([taxable<30000,taxable<80000,taxable<160000,taxable>=160000],[.08,.13,.18,.23])
credits=np.clip(deps*rng.normal(1450,350,n)+rng.normal(400,500,n),0,7000)
liability=np.maximum(0,taxable*rate-credits)
withhold_rate=np.clip(rng.normal(.145,.035,n),.05,.26)
withholding=wages*withhold_rate
estimated=np.where((other>7000)|(rng.random(n)<.12), np.clip(other*rng.uniform(.05,.20,n),0,9000),0)
expenses=np.clip(agi*rng.normal(.70,.12,n),8000,agi*1.05)
savings=np.clip((agi-expenses)/np.maximum(agi,1),-.10,.55)
income_consistency=np.clip(rng.normal(.86,.10,n),.40,1.0)
recurring_share=np.clip(rng.normal(.58,.12,n),.20,.90)
monthly_var=np.clip((1-income_consistency)+rng.normal(.03,.03,n),.01,.65)
# add small unobserved shock so target not deterministic from predictors
true_balance=withholding+estimated-liability+rng.normal(0,2600,n)
outcome=np.where(true_balance>=0,'Refund','Tax Owed')

df=pd.DataFrame({'annual_wages':wages.round(2),'other_income':other.round(2),'adjusted_gross_income':agi.round(2),'filing_status':filing,'dependents':deps,'itemized_deduction':itemized,'deduction_amount':deduction.round(2),'tax_credits':credits.round(2),'federal_withholding':withholding.round(2),'estimated_tax_payments':estimated.round(2),'estimated_tax_liability':liability.round(2),'annual_expenses':expenses.round(2),'savings_rate':savings.round(4),'income_consistency':income_consistency.round(4),'recurring_expense_share':recurring_share.round(4),'monthly_income_variability':monthly_var.round(4),'tax_outcome':outcome})
# engineer predictors available before final outcome; avoid balance/refund amount
for col in ['federal_withholding','estimated_tax_payments']:
    pass
df['withholding_to_income']=df.federal_withholding/df.adjusted_gross_income.clip(lower=1)
df['estimated_payment_to_income']=df.estimated_tax_payments/df.adjusted_gross_income.clip(lower=1)
df['income_to_expense']=df.adjusted_gross_income/df.annual_expenses.clip(lower=1)
df['tax_payments_to_liability']=(df.federal_withholding+df.estimated_tax_payments)/df.estimated_tax_liability.clip(lower=500)
df.to_csv(OUT+'/synthetic_tax_profiles.csv',index=False)

features=['annual_wages','other_income','adjusted_gross_income','filing_status','dependents','itemized_deduction','deduction_amount','tax_credits','federal_withholding','estimated_tax_payments','estimated_tax_liability','annual_expenses','savings_rate','income_consistency','recurring_expense_share','monthly_income_variability','withholding_to_income','estimated_payment_to_income','income_to_expense','tax_payments_to_liability']
X=df[features]; y=(df.tax_outcome=='Refund').astype(int)
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,random_state=42,stratify=y)
cat=['filing_status']; num=[c for c in features if c not in cat]
pre=ColumnTransformer([('num',StandardScaler(),num),('cat',OneHotEncoder(handle_unknown='ignore'),cat)])
models={
'Logistic Regression':LogisticRegression(max_iter=2000,class_weight='balanced'),
'Decision Tree':DecisionTreeClassifier(max_depth=6,min_samples_leaf=30,class_weight='balanced',random_state=42),
'Random Forest':RandomForestClassifier(n_estimators=350,max_depth=10,min_samples_leaf=8,class_weight='balanced',random_state=42,n_jobs=-1)
}
rows=[]; fitted={}
cv=StratifiedKFold(5,shuffle=True,random_state=42)
for name,m in models.items():
    pipe=Pipeline([('pre',pre),('model',m)])
    cvs=cross_val_score(pipe,Xtr,ytr,cv=cv,scoring='roc_auc')
    pipe.fit(Xtr,ytr); fitted[name]=pipe
    pred=pipe.predict(Xte); prob=pipe.predict_proba(Xte)[:,1]
    rows.append([name,accuracy_score(yte,pred),precision_score(yte,pred),recall_score(yte,pred),f1_score(yte,pred),roc_auc_score(yte,prob),cvs.mean(),cvs.std()])
metrics=pd.DataFrame(rows,columns=['Model','Accuracy','Precision','Recall','F1','Test ROC-AUC','CV ROC-AUC Mean','CV ROC-AUC SD'])
metrics.to_csv(OUT+'/model_metrics.csv',index=False)
rf=fitted['Random Forest']; pred=rf.predict(Xte); prob=rf.predict_proba(Xte)[:,1]
cm=confusion_matrix(yte,pred)
np.savetxt(OUT+'/confusion_matrix.csv',cm,fmt='%d',delimiter=',')
# permutation importance on original columns
pi=permutation_importance(rf,Xte,yte,n_repeats=8,random_state=42,scoring='roc_auc',n_jobs=-1)
imp=pd.DataFrame({'Feature':features,'Importance':pi.importances_mean}).sort_values('Importance',ascending=False)
imp.to_csv(OUT+'/feature_importance.csv',index=False)
# subgroup metrics
te=Xte.copy(); te['y']=yte.values; te['pred']=pred
te['income_group']=pd.cut(te.adjusted_gross_income,[0,50000,100000,200000,np.inf],labels=['<$50k','$50k-$100k','$100k-$200k','$200k+'])
subs=[]
for group_col in ['income_group','filing_status']:
  for g,d in te.groupby(group_col,observed=True):
    subs.append([group_col,str(g),len(d),accuracy_score(d.y,d.pred),recall_score(d.y,d.pred,zero_division=0)])
sub=pd.DataFrame(subs,columns=['Group Type','Group','N','Accuracy','Refund Recall']); sub.to_csv(OUT+'/subgroup_metrics.csv',index=False)

plt.figure(figsize=(7,4.2)); df.tax_outcome.value_counts().reindex(['Refund','Tax Owed']).plot(kind='bar'); plt.title('Synthetic Tax Outcome Distribution'); plt.ylabel('Records'); plt.xticks(rotation=0); plt.tight_layout(); plt.savefig(OUT+'/fig1_outcome_distribution.png',dpi=180); plt.close()
plt.figure(figsize=(7,4.2)); top=imp.head(10).sort_values('Importance'); plt.barh(top.Feature,top.Importance); plt.title('Random Forest Permutation Importance'); plt.xlabel('Decrease in ROC-AUC when permuted'); plt.tight_layout(); plt.savefig(OUT+'/fig2_feature_importance.png',dpi=180); plt.close()
plt.figure(figsize=(6,5));
for name,pipe in fitted.items(): RocCurveDisplay.from_estimator(pipe,Xte,yte,name=name,ax=plt.gca())
plt.title('ROC Curves on Held-Out Test Set'); plt.tight_layout(); plt.savefig(OUT+'/fig3_roc_curves.png',dpi=180); plt.close()
plt.figure(figsize=(5,4.2)); plt.imshow(cm); plt.title('Random Forest Confusion Matrix'); plt.xticks([0,1],['Tax Owed','Refund']); plt.yticks([0,1],['Tax Owed','Refund']); plt.xlabel('Predicted'); plt.ylabel('Actual');
for i in range(2):
 for j in range(2): plt.text(j,i,str(cm[i,j]),ha='center',va='center')
plt.tight_layout(); plt.savefig(OUT+'/fig4_confusion_matrix.png',dpi=180); plt.close()
# subgroup accuracy chart
sg=sub[sub['Group Type']=='income_group']
plt.figure(figsize=(7,4.2)); plt.bar(sg.Group,sg.Accuracy); plt.ylim(.5,1); plt.title('Random Forest Accuracy by Income Group'); plt.ylabel('Accuracy'); plt.tight_layout(); plt.savefig(OUT+'/fig5_subgroup_accuracy.png',dpi=180); plt.close()

summary={'n':n,'refund_rate':float(y.mean()),'metrics':metrics.round(4).to_dict('records'),'cm':cm.tolist(),'top_features':imp.head(8).round(4).to_dict('records'),'subgroups':sub.round(4).to_dict('records')}
json.dump(summary,open(OUT+'/summary.json','w'),indent=2)
print(json.dumps(summary,indent=2))


## Reproducibility
Run the code cell from top to bottom in a Python environment with numpy, pandas, matplotlib, and scikit-learn installed. A fixed random seed is used so the synthetic dataset and reported metrics can be reproduced.